# TargetRecon — Python API Demo

This notebook walks through the full TargetRecon Python API:
- Fetching a target report
- Exploring UniProt, PDB, AlphaFold, ChEMBL, and BindingDB data
- Filtering and ranking ligands
- Exporting reports (HTML, JSON, SDF)
- Batch processing multiple targets

**Install:** `pip install targetrecon`

## 1. Installation check

In [1]:
import targetrecon
print("TargetRecon version:", targetrecon.__version__)

TargetRecon version: 0.1.0


## 2. Run a recon — single target

`targetrecon.recon()` accepts a **gene name**, **UniProt accession**, or **ChEMBL target ID**.

It fetches data from all 5 sources in parallel and returns a `TargetReport` object.

In [2]:
report = targetrecon.recon("EGFR")
print(f"Query resolved to: {report.uniprot.uniprot_id} / {report.uniprot.gene_name}")
print(f"Protein name    : {report.uniprot.protein_name}")
print(f"PDB structures  : {report.num_pdb_structures}")
print(f"Bioactivities   : {report.num_bioactivities}")
print(f"Unique ligands  : {report.num_unique_ligands}")

Resolving identifiers for 'EGFR'...

UniProt: P00533  |  ChEMBL: CHEMBL203

Fetching data from 5 sources in parallel...

Done!  375 structures · 10000 bioactivities · 750 unique ligands

Query resolved to: P00533 / EGFR
Protein name    : Epidermal growth factor receptor
PDB structures  : 375
Bioactivities   : 10000
Unique ligands  : 750


### Works with UniProt accessions and ChEMBL IDs too

In [ ]:
# By UniProt accession
report_up = targetrecon.recon("P00533")
print("UniProt input →", report_up.uniprot.gene_name)

# By ChEMBL target ID
report_ch = targetrecon.recon("CHEMBL203")
print("ChEMBL input  →", report_ch.uniprot.gene_name)

## 3. Explore UniProt data

In [ ]:
u = report.uniprot

print("Gene name      :", u.gene_name)
print("Protein name   :", u.protein_name)
print("Organism       :", u.organism)
print("UniProt ID     :", u.uniprot_id)
print("ChEMBL target  :", u.chembl_id)
print("Sequence length:", len(u.sequence) if u.sequence else "N/A")
print("\nFunction (first 300 chars):")
print(u.function[:300] if u.function else "N/A")

In [ ]:
# Subcellular location
print("Subcellular locations:", u.subcellular_locations)

# Diseases
print("\nAssociated diseases:")
for d in u.diseases[:5]:
    print(" -", d)

In [ ]:
# GO terms grouped by category
from collections import defaultdict

go_by_cat = defaultdict(list)
for go in u.go_terms:
    go_by_cat[go.category].append(go.term)

for cat, terms in go_by_cat.items():
    print(f"\nGO — {cat} ({len(terms)} terms):")
    for t in terms[:5]:
        print("  ", t)

## 4. Explore PDB structures

In [ ]:
import pandas as pd

pdb_rows = []
for s in report.pdb_structures:
    pdb_rows.append({
        "PDB ID": s.pdb_id,
        "Method": s.method.value if s.method else "",
        "Resolution (Å)": s.resolution,
        "Deposit Date": s.release_date,
        "Ligands": ", ".join(l.ligand_id for l in s.ligands) if s.ligands else "",
        "Title": (s.title or "")[:60],
    })

pdb_df = pd.DataFrame(pdb_rows)
print(f"{len(pdb_df)} structures")
pdb_df.head(10)

In [ ]:
# Method breakdown
print("Method breakdown:")
print(pdb_df["Method"].value_counts().to_string())

# Resolution statistics
res = pdb_df["Resolution (Å)"].dropna()
print(f"\nResolution stats (Å): min={res.min():.2f}  median={res.median():.2f}  max={res.max():.2f}")

## 5. AlphaFold predicted structure

In [ ]:
af = report.alphafold
if af:
    print("AlphaFold entry     :", af.entry_id)
    print("Model version       :", af.model_version)
    print("PDB download URL    :", af.pdb_url)
    print("Mean pLDDT          :", af.mean_plddt)
else:
    print("No AlphaFold entry found.")

## 6. Bioactivity data (ChEMBL + BindingDB)

In [ ]:
bio_rows = []
for b in report.bioactivities:
    bio_rows.append({
        "Molecule": b.molecule_chembl_id or b.name or "",
        "Activity Type": b.activity_type,
        "Value (nM)": b.value,
        "pChEMBL": b.pchembl_value,
        "Source": b.source,
        "SMILES": (b.smiles or "")[:40],
    })

bio_df = pd.DataFrame(bio_rows)
print(f"{len(bio_df)} bioactivity records")
bio_df.head(10)

In [ ]:
# Source breakdown
print("Source breakdown:")
print(bio_df["Source"].value_counts().to_string())

# Activity type breakdown
print("\nActivity type breakdown:")
print(bio_df["Activity Type"].value_counts().head(8).to_string())

# pChEMBL distribution
pc = bio_df["pChEMBL"].dropna()
print(f"\npChEMBL stats: min={pc.min():.2f}  median={pc.median():.2f}  max={pc.max():.2f}")
print(f"High-potency (pChEMBL ≥ 9): {(pc >= 9).sum()} records")

## 7. Unique ligands ranked by potency

In [ ]:
lig_rows = []
for l in report.ligand_summary:
    lig_rows.append({
        "Name": l.name or "",
        "ChEMBL ID": l.chembl_id or "",
        "Best pChEMBL": l.best_pchembl,
        "Activity Type": l.best_activity_type,
        "Activity (nM)": l.best_activity_value_nM,
        "# Assays": l.num_assays,
        "Sources": ", ".join(l.sources),
        "SMILES": (l.smiles or "")[:50],
    })

lig_df = pd.DataFrame(lig_rows)
print(f"{len(lig_df)} unique ligands (sorted by pChEMBL descending)")
lig_df.head(20)

In [ ]:
# Best ligand shortcut
best = report.best_ligand
if best:
    print("Best ligand:")
    print(f"  Name     : {best.name}")
    print(f"  ChEMBL ID: {best.chembl_id}")
    print(f"  pChEMBL  : {best.best_pchembl}")
    print(f"  SMILES   : {best.smiles}")

In [ ]:
# Filter ligands programmatically
potent = [l for l in report.ligand_summary if l.best_pchembl and l.best_pchembl >= 9.0]
print(f"Ligands with pChEMBL ≥ 9.0: {len(potent)}")

ic50_only = [l for l in report.ligand_summary if l.best_activity_type == "IC50"]
print(f"Ligands measured by IC50: {len(ic50_only)}")

multi_source = [l for l in report.ligand_summary if len(l.sources) > 1]
print(f"Ligands confirmed in multiple databases: {len(multi_source)}")

## 8. Protein–protein interactions (STRING DB)

In [ ]:
if report.interactions:
    int_rows = []
    for i in report.interactions:
        int_rows.append({
            "Partner": i.partner_gene,
            "Score": i.score,
            "Interaction Type": i.interaction_type,
        })
    int_df = pd.DataFrame(int_rows).sort_values("Score", ascending=False)
    print(f"{len(int_df)} protein interactions")
    print(int_df.to_string(index=False))
else:
    print("No interaction data.")

## 9. Export reports

In [ ]:
from targetrecon.core import save_html, save_json, save_sdf
from pathlib import Path

out = Path("outputs")
out.mkdir(exist_ok=True)

# HTML — interactive self-contained report
html_path = save_html(report, out / "EGFR_report.html")
print("HTML report  →", html_path)

# JSON — full machine-readable report
json_path = save_json(report, out / "EGFR_report.json")
print("JSON report  →", json_path)

# SDF — top 20 ligands with 3D conformers (RDKit MMFF)
sdf_path = save_sdf(report, out / "EGFR_top20_ligands.sdf", top_n=20)
print("SDF ligands  →", sdf_path)

In [ ]:
# SDF with filters — only IC50, pChEMBL ≥ 8, top 50
sdf_filtered = save_sdf(
    report,
    out / "EGFR_IC50_filtered.sdf",
    top_n=50,
    min_pchembl=8.0,
    activity_type="IC50",
)
print("Filtered SDF →", sdf_filtered)

## 10. Batch processing — compare targets

Fetch multiple targets concurrently using `asyncio.gather()` via `recon_async()`.

In [ ]:
import asyncio

panel = ["EGFR", "BRAF", "CDK2", "ABL1"]

panel_reports = await asyncio.gather(*[
    targetrecon.recon_async(t, verbose=False) for t in panel
])

summary = []
for t, r in zip(panel, panel_reports):
    if r.uniprot is None:
        continue
    summary.append({
        "Target": t,
        "UniProt": r.uniprot.uniprot_id,
        "PDB Structures": r.num_pdb_structures,
        "Bioactivities": r.num_bioactivities,
        "Unique Ligands": r.num_unique_ligands,
        "Best pChEMBL": r.best_ligand.best_pchembl if r.best_ligand else None,
        "Best Ligand": r.best_ligand.name or r.best_ligand.chembl_id if r.best_ligand else None,
    })

pd.DataFrame(summary)

In [ ]:
# Export all panel reports as HTML
for t, r in zip(panel, panel_reports):
    if r.uniprot:
        p = save_html(r, out / f"{t}_report.html")
        print(f"  {t} → {p}")

## 11. Work with raw JSON

The `TargetReport` is a Pydantic model — serialize/deserialize freely.

In [ ]:
# Serialize to dict
data = report.model_dump()
print("Top-level keys:", list(data.keys()))

# Serialize to JSON string
json_str = report.model_dump_json(indent=2)
print(f"\nJSON size: {len(json_str):,} characters")

# Load back from JSON file
from targetrecon.models import TargetReport
with open(out / "EGFR_report.json") as f:
    loaded = TargetReport.model_validate_json(f.read())
print(f"\nRound-trip: {loaded.uniprot.gene_name} | {loaded.num_unique_ligands} ligands")

## 12. Quick visualization (optional — requires matplotlib)

In [ ]:
try:
    import matplotlib.pyplot as plt
    from collections import Counter

    pchembl_vals = [b.pchembl_value for b in report.bioactivities if b.pchembl_value]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # pChEMBL histogram
    axes[0].hist(pchembl_vals, bins=30, color="#58a6ff", edgecolor="white", linewidth=0.5)
    axes[0].axvline(7, color="#f85149", linestyle="--", label="pChEMBL = 7 (100 nM)")
    axes[0].axvline(9, color="#3fb950", linestyle="--", label="pChEMBL = 9 (1 nM)")
    axes[0].set_xlabel("pChEMBL value")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"EGFR — pChEMBL distribution (n={len(pchembl_vals)})")
    axes[0].legend()

    # Activity type bar chart
    atype_counts = Counter(b.activity_type for b in report.bioactivities if b.activity_type)
    top_types = dict(atype_counts.most_common(6))
    axes[1].bar(top_types.keys(), top_types.values(), color="#bc8cff", edgecolor="white")
    axes[1].set_xlabel("Activity type")
    axes[1].set_ylabel("Count")
    axes[1].set_title("EGFR — Bioactivity type breakdown")

    plt.tight_layout()
    plt.savefig(out / "EGFR_charts.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Chart saved →", out / "EGFR_charts.png")

except ImportError:
    print("matplotlib not installed — pip install matplotlib to enable visualizations")